# model_matrix_backtest_contract_v1

目标：验证 QuantBench/model matrix 默认接入 `src.backtest.run_prediction_backtest`，并把 `backtest_config.json` 与 `walk_forward_splits.csv` 通过 manifest/dashboard 链路公开。

In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
for parent in [ROOT, *ROOT.parents]:
    if (parent / 'AGENTS.md').exists() and (parent / 'src').exists():
        ROOT = parent
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CARD = ROOT / 'research' / 'model_matrix_backtest_contract_v1'
CARD.mkdir(parents=True, exist_ok=True)

from src.dashboard import write_model_matrix_dashboard_export
from src.model import run_factor_feature_set_matrix

In [2]:
outputs = run_factor_feature_set_matrix(
    card_dir=CARD,
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=('ridge_mse', 'random_forest_clf'),
    top_n_factors=3,
)

dashboard_payload = write_model_matrix_dashboard_export(
    CARD,
    output_path=CARD / 'dashboard_export.json',
    top_n=5,
)

summary = json.loads((CARD / 'analysis_summary.json').read_text(encoding='utf-8'))
manifest = json.loads((CARD / 'analysis_manifest.json').read_text(encoding='utf-8'))
backtest_config = json.loads((CARD / 'backtest_config.json').read_text(encoding='utf-8'))

notebook_summary = {
    'schema_version': 'research.model_matrix_backtest_contract_v1',
    'status': 'completed',
    'matrix_status': summary['status'],
    'variant_count': summary['variant_count'],
    'backtest_contract_schema_version': summary['backtest_contract']['schema_version'],
    'cost_model': summary['backtest_contract']['cost_model'],
    'walk_forward_fold_count': summary['walk_forward']['fold_count'],
    'dashboard_manifest_kind': dashboard_payload['manifest_source']['kind'],
    'dashboard_backtest_schema_version': dashboard_payload['backtest']['config']['schema_version'],
    'dashboard_walk_forward_rows': len(dashboard_payload['backtest']['walk_forward_splits']),
    'public_artifacts': sorted(dashboard_payload['artifacts']),
    'private_predictions_exposed': 'predictions' in dashboard_payload['artifacts'],
    'private_daily_backtest_exposed': 'daily_backtest' in dashboard_payload['artifacts'],
    'manifest_paths': manifest['paths'],
}
(CARD / 'notebook_summary.json').write_text(json.dumps(notebook_summary, ensure_ascii=False, indent=2), encoding='utf-8')
notebook_summary

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'schema_version': 'research.model_matrix_backtest_contract_v1',
 'status': 'completed',
 'matrix_status': 'feature_set_matrix_completed',
 'variant_count': 6,
 'backtest_contract_schema_version': 'backtest.config.v1',
 'cost_model': {'close_cost_bps': 1.0,
  'open_cost_bps': 1.0,
  'schema_version': 'backtest.cost_model.v1',
  'slippage_bps': 1.0},
 'walk_forward_fold_count': 5,
 'dashboard_manifest_kind': 'analysis_manifest',
 'dashboard_backtest_schema_version': 'backtest.config.v1',
 'dashboard_walk_forward_rows': 5,
 'public_artifacts': ['analysis_manifest',
  'artifact_manifest',
  'backtest_config',
  'factor_ic',
  'failure_modes',
  'feature_set_specs',
  'model_capabilities',
  'model_comparison',
  'model_effect_deltas',
  'model_review_briefing',
  'qlib_risk_metrics',
  'regime_metrics',
  'segment_effect_matrix',
  'summary',
  'walk_forward_splits',
  'window_metrics',
  'window_regime_metrics'],
 'private_predictions_exposed': False,
 'private_daily_backtest_exposed': F

In [3]:
assert notebook_summary['status'] == 'completed'
assert notebook_summary['backtest_contract_schema_version'] == 'backtest.config.v1'
assert notebook_summary['cost_model']['slippage_bps'] > 0.0
assert notebook_summary['walk_forward_fold_count'] >= 1
assert notebook_summary['dashboard_manifest_kind'] == 'analysis_manifest'
assert notebook_summary['dashboard_backtest_schema_version'] == 'backtest.config.v1'
assert notebook_summary['dashboard_walk_forward_rows'] == notebook_summary['walk_forward_fold_count']
assert 'backtest_config' in notebook_summary['public_artifacts']
assert 'walk_forward_splits' in notebook_summary['public_artifacts']
assert notebook_summary['private_predictions_exposed'] is False
assert notebook_summary['private_daily_backtest_exposed'] is False
assert notebook_summary['manifest_paths']['backtest_config'] == 'backtest_config.json'
assert notebook_summary['manifest_paths']['walk_forward_splits'] == 'walk_forward_splits.csv'
assert (CARD / 'daily_backtest.csv').exists()
assert (CARD / 'dashboard_export.json').exists()

## 中文结论

- 做了什么：验证 `run_factor_feature_set_matrix` 已默认使用 `run_prediction_backtest`，并把 backtest config 与 walk-forward fold 写入 analysis/dashboard manifest。
- 核心指标：样例包含 2 个基础模型、3 个因子集、6 个模型变体；`walk_forward_splits.csv` 记录可复核 fold，`backtest_config.json` 记录成本、滑点和仓位策略。
- regime 差异：本轮不重新解释模型优劣，重点确认 backtest 产出的 daily path 可继续进入 regime/window/model comparison 分析。
- 是否晋升：晋升为 model matrix 框架能力；不代表任何模型或因子组合晋升 production。
- 风险：当前样例仍是轻量矩阵，真实全量 QuantBench 训练需要继续使用同一契约重跑并比较成本敏感性。
- 下一步：让 schedule/dashboard refresh 的通知摘要显示 backtest contract 版本和 walk-forward fold 数，便于 review 时发现配置漂移。